# Task 4: выходное представление и функции потерь

Эксперименты с одной нейросетью на выданных 14 000 строках. Исходный `task4.ipynb` сохранён без изменений. Сеть, оптимизация и метрики реализованы только через PyTorch. NumPy, pandas и sklearn применяются только для загрузки, разбиения и предобработки.

Сравниваем MSE, Huber, Gaussian NLL, ограниченную регрессионную голову, распределение по годам и ordinal CDF. Многочисленные проверки на одном validation могут переобучить выбор; поэтому лидеры проверяются на ещё двух разбиениях development, всегда с исходной residual-сетью в паре. Audit не используется для выбора; после заморозки кандидатов он один раз оценивается в заключительном приложении. Ансамбли и внешние данные отсутствуют.

Точный public score зависит от скрытой разметки. Оптимизируем MSE, поскольку исходный public score 85.55280 согласуется с MSE; локальный RMSE исходного решения был 9.1591. Метрики RMSE и MSE одинаково ранжируют фиксированную выборку.


In [1]:
import base64
import copy
import hashlib
import io
import math
import random
import time
import unittest
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import HTML, Markdown, display
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, RobustScaler, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
SELECTION_METRIC = "rmse"  # "rmse", "mse" или "mae"; см. пояснение выше.
assert SELECTION_METRIC in {"rmse", "mse", "mae"}
MAX_EPOCHS = 160
PATIENCE = 25
BATCH_SIZE = 256
# CPU выбран для воспроизводимого сравнения двух реализаций оптимизатора.
# Для большого запуска можно указать "cuda" или "mps", если устройство доступно.
DEVICE = torch.device("cpu")
torch.set_num_threads(2)


def seed_everything(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)


seed_everything()
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})
print(f"torch={torch.__version__}; numpy={np.__version__}; "
      f"pandas={pd.__version__}; sklearn={sklearn.__version__}; device={DEVICE}")
import json
import statistics
import torch.nn.functional as F
OUTPUT_DIR = Path("/tmp/task4_objectives")
OUTPUT_DIR.mkdir(exist_ok=True)
RESULT_PATH = Path("/tmp/task4_objectives_results.json")


torch=2.14.1; numpy=2.4.6; pandas=3.0.3; sklearn=1.9.0; device=cpu


In [2]:
candidates = [Path("task4_data"), Path("homeworks/task4_data")]
DATA_DIR = next((p for p in candidates if (p / "train_x.csv").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Запустите ноутбук из корня репозитория или из homeworks.")

train_x = pd.read_csv(DATA_DIR / "train_x.csv", index_col=0)
train_y = pd.read_csv(DATA_DIR / "train_y.csv", index_col=0)
test_table = pd.read_csv(DATA_DIR / "test_x.csv")
assert train_x.index.is_unique and train_y.index.is_unique
assert set(train_x.index) == set(train_y.index), "Индексы признаков и меток не совпадают"
train_y = train_y.loc[train_x.index]
assert list(train_y.columns) == ["year"]
assert "id" in test_table and test_table["id"].is_unique
assert not test_table["id"].isna().any()

feature_columns = list(train_x.columns)
assert len(feature_columns) == 90
assert set(test_table.columns) == set(feature_columns) | {"id"}
test_ids = test_table["id"].copy()
test_x = test_table.loc[:, feature_columns].copy()
assert all(pd.api.types.is_numeric_dtype(t) for t in train_x.dtypes)
assert all(pd.api.types.is_numeric_dtype(t) for t in test_x.dtypes)
X = train_x.to_numpy(dtype=np.float64)
X_test = test_x.to_numpy(dtype=np.float64)
y = torch.tensor(train_y["year"].to_numpy(dtype=np.float32))
assert torch.isfinite(y).all() and len(y) == len(X)
assert torch.equal(y, y.round()), "Годы должны быть целыми"
assert not np.isinf(X).any() and not np.isinf(X_test).any()

print(f"Обучение: {X.shape}; тест: {X_test.shape}; целевая переменная: year")
print(f"Пропуски: train={np.isnan(X).sum()}, test={np.isnan(X_test).sum()}")
print(f"Точные дубликаты train: {train_x.duplicated().sum()}")
for name in ("train_x.csv", "train_y.csv", "test_x.csv"):
    print(name, "SHA256:", hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest())

Обучение: (14000, 90); тест: (6000, 90); целевая переменная: year
Пропуски: train=0, test=0
Точные дубликаты train: 0
train_x.csv SHA256: aab2f64a4fdb0fd6af98ed95fedc9a3d9a54506b1f607da725926f654f5bff89
train_y.csv SHA256: 1f72519091e6e08948c430a1db9b181df28095e0cc9f29694570b40daed683bf
test_x.csv SHA256: 54f91d13c6f61fd634bae76bdad109471585a4ce0b308159be16f768134b4046


In [3]:
periods = np.digitize(y.tolist(), bins=[1970, 1980, 1990, 2000, 2005])
indices = np.arange(len(X))
development_idx, audit_idx = train_test_split(indices, test_size=0.15, random_state=42, stratify=periods)
def make_split(seed):
    return train_test_split(development_idx, test_size=2100/11900, random_state=seed,
                            stratify=periods[development_idx])
train_idx, valid_idx = make_split(42)
y_train, y_valid = y[train_idx], y[valid_idx]
assert (len(train_idx), len(valid_idx), len(audit_idx)) == (9800, 2100, 2100)
assert not set(development_idx) & set(audit_idx)
print("Split sizes:", len(train_idx), len(valid_idx), len(audit_idx), "; audit не оценивается")


Split sizes: 9800 2100 2100 ; audit не оценивается


In [4]:
class FeaturePreprocessor:
    def __init__(self, kind="standard"):
        if kind not in {"standard", "asinh", "quantile"}:
            raise ValueError(kind)
        self.kind = kind
        self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)

    def fit(self, x):
        z = self.imputer.fit_transform(x)
        if self.kind == "asinh":
            self.robust = RobustScaler().fit(z)
            z = np.arcsinh(self.robust.transform(z))
        self.scaler = (
            QuantileTransformer(n_quantiles=min(256, len(z)), output_distribution="normal",
                                subsample=None, random_state=SEED)
            if self.kind == "quantile" else StandardScaler()
        )
        self.scaler.fit(z)
        return self

    def transform(self, x):
        z = self.imputer.transform(x)
        if self.kind == "asinh":
            z = np.arcsinh(self.robust.transform(z))
        z = self.scaler.transform(z).astype(np.float32)
        if not np.isfinite(z).all():
            raise ValueError("Нефинитные признаки после преобразования")
        return torch.from_numpy(z.copy())


class TargetScaler:
    def __init__(self, target):
        self.mean = float(target.mean())
        self.std = max(float(target.std(unbiased=False)), 1e-6)

    def encode(self, target):
        return (target - self.mean) / self.std

    def decode(self, prediction):
        return prediction * self.std + self.mean


@torch.inference_mode()
def regression_metrics(prediction, target):
    assert prediction.shape == target.shape
    error = prediction.double() - target.double()
    mse = error.square().mean().item()
    return {"rmse": math.sqrt(mse), "mse": mse, "mae": error.abs().mean().item()}


def print_metrics(label, values):
    print(f"{label}: RMSE={values['rmse']:.4f}; MSE={values['mse']:.4f}; MAE={values['mae']:.4f}")


constant = y_train.median() if SELECTION_METRIC == "mae" else y_train.mean()
constant_valid = regression_metrics(torch.full_like(y_valid, float(constant)), y_valid)
print_metrics("Константа, validation", constant_valid)

Константа, validation: RMSE=10.9058; MSE=118.9360; MAE=8.1448


## Оптимизатор и исходная архитектура
Сохраняем исходные ResidualBlock, YearRegressor и собственный AdamW. Последняя голова новой модели меняется в зависимости от гипотезы. Для исходного MSE используются буквально исходные класс, DataLoader и scheduler с горизонтом 160 эпох. Ограничение screening — 60 эпох, early stopping patience 12; для исходной сети и подтверждения patience 25 и максимум 160. Лучший checkpoint выбирается по validation MSE.


In [5]:
class CustomAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)
        for group in self.param_groups:
            if not math.isfinite(group["lr"]) or group["lr"] < 0:
                raise ValueError("lr должен быть конечным и неотрицательным")
            if not math.isfinite(group["eps"]) or group["eps"] <= 0:
                raise ValueError("eps должен быть конечным и положительным")
            if not math.isfinite(group["weight_decay"]) or group["weight_decay"] < 0:
                raise ValueError("weight_decay должен быть конечным и неотрицательным")
            if len(group["betas"]) != 2 or not all(0 <= b < 1 for b in group["betas"]):
                raise ValueError("betas должны лежать в [0, 1)")

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            beta1, beta2 = group["betas"]
            for parameter in group["params"]:
                grad = parameter.grad
                if grad is None:
                    continue
                if grad.is_sparse or torch.is_complex(parameter):
                    raise RuntimeError("Поддерживаются только плотные вещественные градиенты")
                state = self.state[parameter]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(parameter)
                    state["exp_avg_sq"] = torch.zeros_like(parameter)
                state["step"] += 1
                m, v = state["exp_avg"], state["exp_avg_sq"]
                m.lerp_(grad, 1 - beta1)
                v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                correction1 = 1 - beta1 ** state["step"]
                correction2 = 1 - beta2 ** state["step"]
                denominator = v.sqrt().div_(math.sqrt(correction2)).add_(group["eps"])
                parameter.mul_(1 - group["lr"] * group["weight_decay"])
                parameter.addcdiv_(m, denominator, value=-group["lr"] / correction1)
        return loss

In [6]:
class TestCustomAdamW(unittest.TestCase):
    def test_reference_steps_and_parameter_groups(self):
        generator = torch.Generator().manual_seed(101)
        ours = [nn.Parameter(torch.randn(3, 4, generator=generator, dtype=torch.float64)),
                nn.Parameter(torch.randn(4, generator=generator, dtype=torch.float64))]
        reference = [nn.Parameter(p.detach().clone()) for p in ours]
        def groups(parameters):
            return [{"params": [parameters[0]], "lr": 0.003, "weight_decay": 0.1},
                    {"params": [parameters[1]], "lr": 0.002, "weight_decay": 0.0}]
        custom = CustomAdamW(groups(ours), betas=(0.8, 0.95), eps=1e-6)
        builtin = torch.optim.AdamW(groups(reference), betas=(0.8, 0.95), eps=1e-6,
                                    foreach=False, fused=False)
        for step in range(20):
            for i, (p, q) in enumerate(zip(ours, reference)):
                grad = torch.randn(p.shape, generator=generator, dtype=p.dtype)
                if step % 4 == 0 and i == 1:
                    p.grad = q.grad = None
                else:
                    p.grad = torch.zeros_like(grad) if step == 7 else grad
                    q.grad = p.grad.clone()
            custom.step()
            builtin.step()
            for p, q in zip(ours, reference):
                torch.testing.assert_close(p, q, rtol=1e-11, atol=1e-12)

    def test_none_vs_zero_gradient(self):
        p = nn.Parameter(torch.tensor([2.0], dtype=torch.float64))
        optimizer = CustomAdamW([p], lr=0.1, weight_decay=0.2)
        optimizer.step()
        self.assertEqual(p.item(), 2.0)
        self.assertNotIn(p, optimizer.state)
        p.grad = torch.zeros_like(p)
        optimizer.step()
        self.assertAlmostEqual(p.item(), 1.96)
        self.assertEqual(optimizer.state[p]["step"], 1)

    def test_restore_state(self):
        p = nn.Parameter(torch.tensor([1.0, -2.0], dtype=torch.float64))
        a = CustomAdamW([p])
        p.grad = torch.tensor([0.3, -0.1], dtype=p.dtype)
        a.step()
        q = nn.Parameter(p.detach().clone())
        b = CustomAdamW([q])
        b.load_state_dict(copy.deepcopy(a.state_dict()))
        for _ in range(5):
            p.grad = torch.tensor([-0.2, 0.4], dtype=p.dtype)
            q.grad = p.grad.clone()
            a.step()
            b.step()
        torch.testing.assert_close(p, q, rtol=0, atol=0)

    def test_invalid_settings(self):
        for kwargs in [{"lr": -1}, {"eps": 0}, {"betas": (1, 0.9)}, {"weight_decay": -1}]:
            with self.assertRaises(ValueError):
                CustomAdamW([nn.Parameter(torch.ones(1))], **kwargs)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCustomAdamW)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()

test_invalid_settings (__main__.TestCustomAdamW.test_invalid_settings) ... ok
test_none_vs_zero_gradient (__main__.TestCustomAdamW.test_none_vs_zero_gradient) ... ok
test_reference_steps_and_parameter_groups (__main__.TestCustomAdamW.test_reference_steps_and_parameter_groups) ... ok
test_restore_state (__main__.TestCustomAdamW.test_restore_state) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.013s

OK


In [7]:
class ResidualBlock(nn.Module):
    def __init__(self, width, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(width), nn.Linear(width, width * 2), nn.SiLU(),
            nn.Dropout(dropout), nn.Linear(width * 2, width), nn.Dropout(dropout)
        )
        nn.init.zeros_(self.net[-2].weight)
        nn.init.zeros_(self.net[-2].bias)

    def forward(self, x):
        return x + self.net(x)


class YearRegressor(nn.Module):
    def __init__(self, n_features, architecture="residual", width=128, dropout=0.15):
        super().__init__()
        if architecture == "linear":
            self.net = nn.Linear(n_features, 1)
        elif architecture == "mlp":
            self.net = nn.Sequential(
                nn.Linear(n_features, width), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(width, width), nn.ReLU(), nn.Dropout(dropout), nn.Linear(width, 1)
            )
        elif architecture == "residual":
            self.net = nn.Sequential(
                nn.Linear(n_features, width), nn.SiLU(),
                ResidualBlock(width, dropout), ResidualBlock(width, dropout),
                nn.LayerNorm(width), nn.SiLU(), nn.Linear(width, 1)
            )
        else:
            raise ValueError(architecture)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def build_optimizer(model, kind, lr, weight_decay):
    decay, no_decay = [], []
    for parameter in model.parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    groups = [{"params": decay, "weight_decay": weight_decay},
              {"params": no_decay, "weight_decay": 0.0}]
    if kind == "custom":
        return CustomAdamW(groups, lr=lr)
    if kind == "builtin":
        return torch.optim.AdamW(groups, lr=lr, foreach=False, fused=False)
    raise ValueError(kind)


@torch.inference_mode()
def predict(model, features, target_scaler, batch_size=1024):
    model.eval()
    outputs = [model(batch.to(DEVICE)).cpu() for batch in features.split(batch_size)]
    return target_scaler.decode(torch.cat(outputs))


def make_loader(features, targets, seed):
    return DataLoader(TensorDataset(features, targets), batch_size=BATCH_SIZE, shuffle=True,
                      generator=torch.Generator().manual_seed(seed), num_workers=0)


def train_epoch(model, loader, optimizer):
    model.train()
    loss_fn = nn.L1Loss() if SELECTION_METRIC == "mae" else nn.MSELoss()
    total_loss, total_grad, count = 0.0, 0.0, 0
    for features, targets in loader:
        features, targets = features.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(features)
        assert prediction.shape == targets.shape
        loss = loss_fn(prediction, targets)
        if not torch.isfinite(loss):
            raise RuntimeError("Обучение разошлось")
        loss.backward()
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0,
                                           error_if_nonfinite=True)
        optimizer.step()
        total_loss += loss.item() * len(features)
        total_grad += float(grad_norm) * len(features)
        count += len(features)
    return total_loss / count, total_grad / count


def learning_rate(config, epoch):
    # Та же функция используется при final refit, без изменения горизонта.
    phase = (epoch - 1) / max(1, MAX_EPOCHS - 1)
    return config["lr"] * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * phase)))


def fit_experiment(config, optimizer_kind="builtin", seed=SEED):
    seed_everything(seed)
    preprocess = FeaturePreprocessor(config["preprocessing"]).fit(X[train_idx])
    xt, xv = preprocess.transform(X[train_idx]), preprocess.transform(X[valid_idx])
    target_scaler = TargetScaler(y_train)
    loader = make_loader(xt, target_scaler.encode(y_train), seed)
    model = YearRegressor(xt.shape[1], config["architecture"], config["width"],
                          config["dropout"]).to(DEVICE)
    optimizer = build_optimizer(model, optimizer_kind, config["lr"], config["weight_decay"])
    best_score, best_epoch, best_state, stale = math.inf, 0, None, 0
    history = []
    started = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        lr = learning_rate(config, epoch)
        for group in optimizer.param_groups:
            group["lr"] = lr
        loss, grad_norm = train_epoch(model, loader, optimizer)
        train_metrics = regression_metrics(predict(model, xt, target_scaler), y_train)
        valid_metrics = regression_metrics(predict(model, xv, target_scaler), y_valid)
        score = valid_metrics[SELECTION_METRIC]
        history.append({"epoch": epoch, "loss": loss, "grad_norm": grad_norm, "lr": lr,
                        **{f"train_{k}": v for k, v in train_metrics.items()},
                        **{f"valid_{k}": v for k, v in valid_metrics.items()}})
        if score < best_score:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
        if epoch == 1 or epoch % 20 == 0:
            print(f"{config['name']} / {optimizer_kind}: epoch={epoch:3d}, "
                  f"train RMSE={train_metrics['rmse']:.4f}, valid RMSE={valid_metrics['rmse']:.4f}",
                  flush=True)
        if stale >= PATIENCE:
            break
    assert best_state is not None
    model.load_state_dict(best_state)
    predictions = predict(model, xv, target_scaler)
    result = dict(config=dict(config), optimizer=optimizer_kind, model=model.cpu(),
                  preprocessing=preprocess, target_scaler=target_scaler, history=history,
                  best_epoch=best_epoch, metrics=regression_metrics(predictions, y_valid),
                  valid_prediction=predictions, seconds=time.perf_counter() - started)
    print_metrics(f"Лучший checkpoint ({best_epoch} эпоха)", result["metrics"])
    return result

## Обобщённая выходная голова
Для распределения сеть возвращает вероятности, а прогнозом служит математическое ожидание года; это одна модель. Мягкие метки получаем из Gaussian kernel по опорным годам. Loss сочетает MSE среднего с cross entropy либо ошибкой CDF. Empirical prior вычисляется исключительно по train. Для ordinal головы вероятности `P(year > threshold)` приводятся к невозрастающему виду cumulative minimum; ожидаемый год получается интегрированием survival function. Gaussian NLL моделирует дисперсию, но в submission используется только среднее.


In [8]:
class ObjectiveNetwork(nn.Module):
    def __init__(self, config, training_target):
        super().__init__()
        self.config = dict(config)
        self.kind = config["kind"]
        self.target_scaler = TargetScaler(training_target)
        self.mean = self.target_scaler.mean
        self.std = self.target_scaler.std
        self.low = float(training_target.min()) - config.get("margin", 0)
        self.high = float(training_target.max()) + config.get("margin", 0)
        self.register_buffer("support", torch.linspace(self.low, self.high, config.get("bins", 91)))
        if self.kind in {"distribution", "cdf"}:
            output_dim = len(self.support)
        elif self.kind == "ordinal":
            output_dim = len(self.support) - 1
        elif self.kind == "gaussian":
            output_dim = 2
        else:
            output_dim = 1
        if config.get("backbone", "residual") == "batchnorm":
            width, dropout = config["width"], config.get("bn_dropout", 0.4)
            self.net = nn.Sequential(nn.Linear(90, width), nn.BatchNorm1d(width), nn.ReLU(),
                                     nn.Dropout(dropout), nn.Linear(width, width), nn.BatchNorm1d(width),
                                     nn.ReLU(), nn.Dropout(dropout), nn.Linear(width, 1))
        else:
            self.net = YearRegressor(90, "residual", config["width"], config["dropout"]).net
        if output_dim != 1:
            self.net[-1] = nn.Linear(config["width"], output_dim)
        if self.kind in {"distribution", "cdf"} and config.get("prior", False):
            nearest = (training_target[:,None] - self.support[None,:]).abs().argmin(1)
            counts = torch.bincount(nearest, minlength=len(self.support)).float() + 0.5
            self.register_buffer("prior_logits", (counts/counts.sum()).log())
        else:
            self.register_buffer("prior_logits", torch.zeros(len(self.support)))

    def forward(self, features):
        raw = self.net(features)
        if self.kind in {"mse", "huber"}:
            z = raw[:,0]
        elif self.kind == "bounded":
            p = (self.mean-self.low)/(self.high-self.low)
            p = min(max(p, 1e-5), 1-1e-5)
            logit = math.log(p/(1-p))
            year = self.low + (self.high-self.low)*torch.sigmoid(raw[:,0]+logit)
            z = (year-self.mean)/self.std
        elif self.kind == "upper_softplus":
            offset = math.log(math.expm1(max((self.high-self.mean)/self.std, 1e-3)))
            z = (self.high-self.mean)/self.std-F.softplus(raw[:,0]+offset)
        elif self.kind == "gaussian":
            z = raw[:,0]
        elif self.kind in {"distribution", "cdf"}:
            probs = (raw+self.prior_logits).softmax(1)
            z = ((probs*self.support).sum(1)-self.mean)/self.std
        elif self.kind == "ordinal":
            probs = torch.sigmoid(raw)
            monotone = torch.cummin(probs, dim=1).values
            step = (self.high-self.low)/(len(self.support)-1)
            z = (self.low+step*monotone.sum(1)-self.mean)/self.std
        else:
            raise ValueError(self.kind)
        return z, raw

    def loss(self, prediction, raw, target):
        mse = F.mse_loss(prediction, target)
        c = self.config
        if self.kind == "huber":
            return F.huber_loss(prediction, target, delta=c["delta"])
        if self.kind == "gaussian":
            logvar = raw[:,1].clamp(-3, 3)
            nll = 0.5*((prediction-target).square()*torch.exp(-logvar)+logvar).mean()
            return c["mse_weight"]*mse+c["aux_weight"]*nll
        if self.kind in {"distribution", "cdf"}:
            years = target*self.std+self.mean
            sigma = c.get("sigma", 1.0)
            target_probs = (-0.5*((years[:,None]-self.support[None,:])/sigma).square()).softmax(1)
            logp = (raw+self.prior_logits).log_softmax(1)
            if self.kind == "distribution":
                auxiliary = -(target_probs*logp).sum(1).mean()
            else:
                auxiliary = F.mse_loss(logp.exp().cumsum(1), target_probs.cumsum(1))
            return c["mse_weight"]*mse+c["aux_weight"]*auxiliary
        if self.kind == "ordinal":
            years = target*self.std+self.mean
            thresholds = 0.5*(self.support[:-1]+self.support[1:])
            target_survival = (years[:,None]>thresholds[None,:]).float()
            auxiliary = F.binary_cross_entropy_with_logits(raw, target_survival)
            return c["mse_weight"]*mse+c["aux_weight"]*auxiliary
        return mse

@torch.inference_mode()
def objective_predict(model, features):
    model.eval()
    z = torch.cat([model(batch)[0] for batch in features.split(1024)])
    return z*model.std+model.mean

def run_trial(config, seed, stage="screen", optimizer_kind="builtin"):
    seed_everything(seed)
    tr, va = make_split(seed)
    pp = FeaturePreprocessor(config.get("preprocessing", "standard")).fit(X[tr])
    xt, xv = pp.transform(X[tr]), pp.transform(X[va])
    target_scaler = TargetScaler(y[tr])
    loader = make_loader(xt, target_scaler.encode(y[tr]), seed)
    model = ObjectiveNetwork(config, y[tr])
    optimizer = build_optimizer(model, optimizer_kind, config["lr"], config["weight_decay"])
    best_score, best_epoch, best_state, stale = math.inf, 0, None, 0
    start = time.perf_counter()
    history = []
    patience = 25 if config["id"] == "baseline" or stage == "confirmation" else 12
    max_epochs = 160 if config["id"] == "baseline" or stage == "confirmation" else 60
    for epoch in range(1, max_epochs+1):
        model.train()
        for group in optimizer.param_groups:
            group["lr"] = learning_rate(config, epoch)
        loss_sum = grad_sum = 0.0
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            pred, raw = model(xb)
            loss = model.loss(pred, raw, yb)
            if not torch.isfinite(loss):
                raise RuntimeError((config, seed, epoch, "nonfinite loss"))
            loss.backward()
            grad = nn.utils.clip_grad_norm_(model.parameters(), 5.0, error_if_nonfinite=True)
            optimizer.step()
            loss_sum += float(loss.detach())*len(xb)
            grad_sum += float(grad)*len(xb)
        vp = objective_predict(model, xv)
        metrics = regression_metrics(vp, y[va])
        history.append(dict(epoch=epoch, loss=loss_sum/len(tr), grad_norm=grad_sum/len(tr), **metrics))
        if metrics["mse"] < best_score:
            best_score, best_epoch, stale = metrics["mse"], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if stale >= patience:
            break
    model.load_state_dict(best_state)
    vp = objective_predict(model, xv)
    metrics = regression_metrics(vp, y[va])
    result = dict(config=dict(config), seed=seed, stage=stage, optimizer=optimizer_kind, best_epoch=best_epoch,
                  stopped_epoch=epoch, seconds=time.perf_counter()-start, history=history, **metrics)
    torch.save(dict(state=best_state, config=dict(config), seed=seed, best_epoch=best_epoch,
                    valid_idx=torch.tensor(va), train_idx=torch.tensor(tr), valid_prediction=vp,
                    target_mean=model.mean, target_std=model.std, **metrics),
               OUTPUT_DIR / (f'{config["id"]}_seed{seed}' + ("_custom" if optimizer_kind == "custom" else "") + ".pt"))
    print(f'{stage}: {config["id"]}, seed={seed}, MSE={metrics["mse"]:.6f}, '
          f'RMSE={metrics["rmse"]:.6f}, epoch={best_epoch}, {result["seconds"]:.1f}s', flush=True)
    return result


In [9]:
common = dict(width=128, dropout=0.15, lr=1e-3, weight_decay=1e-2, preprocessing="standard")
configs = [dict(common, id="baseline", kind="mse")]
for delta in [0.75, 1.5, 2.5, 4.0]:
    configs.append(dict(common, id=f"huber_d{delta:g}", kind="huber", delta=delta))
for margin in [0, 5, 15]:
    configs.append(dict(common, id=f"bounded_m{margin}", kind="bounded", margin=margin))
configs.append(dict(common, id="upper_softplus", kind="upper_softplus"))
for mw, aw in [(0, 1), (1, 0.25), (1, 1)]:
    configs.append(dict(common, id=f"gaussian_m{mw}_n{aw}", kind="gaussian", mse_weight=mw, aux_weight=aw))
for sigma in [0.6, 2, 5]:
    for aw in [0.1, 1]:
        configs.append(dict(common, id=f"dist_s{sigma}_ce{aw}", kind="distribution", bins=91,
                            sigma=sigma, prior=True, mse_weight=1, aux_weight=aw))
for prior in [False, True]:
    configs.append(dict(common, id=f"dist_pure_ce_p{int(prior)}", kind="distribution", bins=91,
                        sigma=2, prior=prior, mse_weight=0, aux_weight=1))
for bins in [16, 32]:
    for aw in [0.1, 1]:
        configs.append(dict(common, id=f"coarse_b{bins}_ce{aw}", kind="distribution", bins=bins,
                            sigma=3, prior=True, mse_weight=1, aux_weight=aw))
for bins in [20, 45]:
    for aw in [0.1, 1]:
        configs.append(dict(common, id=f"ordinal_b{bins}_bce{aw}", kind="ordinal", bins=bins,
                            mse_weight=1, aux_weight=aw))
for aw in [1, 10, 30]:
    configs.append(dict(common, id=f"cdf_aux{aw}", kind="cdf", bins=91, sigma=2, prior=True,
                        mse_weight=1, aux_weight=aw))
configs.append(dict(common, id="dist_mean_only", kind="distribution", bins=91, sigma=2,
                    prior=True, mse_weight=1, aux_weight=0))
assert len(configs) >= 25 and len({c["id"] for c in configs}) == len(configs)
print("Конфигураций:", len(configs))


Конфигураций: 32


In [10]:
results = []
for config in configs:
    results.append(run_trial(config, 42, stage="screen"))
    RESULT_PATH.write_text(json.dumps(results, indent=2))
print("\nВсе результаты seed42:")
for r in sorted(results, key=lambda r: r["mse"]):
    print(f'{r["config"]["id"]:30s} MSE={r["mse"]:.5f} RMSE={r["rmse"]:.5f} epoch={r["best_epoch"]}')


screen: baseline, seed=42, MSE=83.889076, RMSE=9.159098, epoch=4, 2.4s
screen: huber_d0.75, seed=42, MSE=85.250612, RMSE=9.233126, epoch=5, 1.4s
screen: huber_d1.5, seed=42, MSE=83.798203, RMSE=9.154136, epoch=3, 1.2s
screen: huber_d2.5, seed=42, MSE=83.672481, RMSE=9.147266, epoch=3, 1.3s
screen: huber_d4, seed=42, MSE=83.767990, RMSE=9.152485, epoch=4, 1.3s
screen: bounded_m0, seed=42, MSE=84.732509, RMSE=9.205026, epoch=3, 1.2s
screen: bounded_m5, seed=42, MSE=85.169339, RMSE=9.228724, epoch=3, 1.3s
screen: bounded_m15, seed=42, MSE=85.903283, RMSE=9.268402, epoch=3, 1.2s
screen: upper_softplus, seed=42, MSE=83.536682, RMSE=9.139840, epoch=3, 1.2s
screen: gaussian_m0_n1, seed=42, MSE=82.715143, RMSE=9.094787, epoch=4, 1.5s
screen: gaussian_m1_n0.25, seed=42, MSE=82.999500, RMSE=9.110406, epoch=4, 1.3s
screen: gaussian_m1_n1, seed=42, MSE=82.743088, RMSE=9.096323, epoch=4, 1.3s
screen: dist_s0.6_ce0.1, seed=42, MSE=82.113890, RMSE=9.061671, epoch=4, 1.5s
screen: dist_s0.6_ce1, seed=4

In [11]:
baseline_config = configs[0]
leader_ids = [r["config"]["id"] for r in sorted(results, key=lambda r: r["mse"]) if r["config"]["id"] != "baseline"][:3]
leader_configs = [next(c for c in configs if c["id"] == name) for name in leader_ids]
confirmation = []
for seed in [17, 83]:
    for config in [baseline_config] + leader_configs:
        confirmation.append(run_trial(config, seed, stage="confirmation"))
        RESULT_PATH.write_text(json.dumps(results + confirmation, indent=2))
all_runs = results + confirmation
summary = []
for config in [baseline_config] + leader_configs:
    runs = [r for r in all_runs if r["config"]["id"] == config["id"]]
    assert len(runs) == 3
    baseline_runs = {r["seed"]: r for r in all_runs if r["config"]["id"] == "baseline"}
    mean_mse = sum(r["mse"] for r in runs)/3
    differences = [r["mse"]-baseline_runs[r["seed"]]["mse"] for r in runs]
    summary.append(dict(id=config["id"], mean_mse=mean_mse, mean_delta=sum(differences)/3,
                        seeds=[r["seed"] for r in runs], mse=[r["mse"] for r in runs],
                        best_epochs=[r["best_epoch"] for r in runs]))
summary.sort(key=lambda r:r["mean_mse"])
print(json.dumps(summary, indent=2))
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))


confirmation: baseline, seed=17, MSE=79.643057, RMSE=8.924296, epoch=4, 2.4s
confirmation: dist_s0.6_ce1, seed=17, MSE=77.597975, RMSE=8.808971, epoch=5, 2.7s
confirmation: cdf_aux30, seed=17, MSE=79.313111, RMSE=8.905791, epoch=3, 2.8s
confirmation: cdf_aux10, seed=17, MSE=79.802217, RMSE=8.933209, epoch=4, 2.9s
confirmation: baseline, seed=83, MSE=87.147965, RMSE=9.335307, epoch=3, 2.3s
confirmation: dist_s0.6_ce1, seed=83, MSE=85.586595, RMSE=9.251302, epoch=4, 2.7s
confirmation: cdf_aux30, seed=83, MSE=85.767341, RMSE=9.261066, epoch=2, 2.5s
confirmation: cdf_aux10, seed=83, MSE=85.599907, RMSE=9.252022, epoch=4, 2.8s
[
  {
    "id": "dist_s0.6_ce1",
    "mean_mse": 81.4851664940871,
    "mean_delta": -2.0748663716302027,
    "seeds": [
      42,
      17,
      83
    ],
    "mse": [
      81.27092941213222,
      77.597974909026,
      85.58659516110306
    ],
    "best_epochs": [
      4,
      5,
      4
    ]
  },
  {
    "id": "cdf_aux30",
    "mean_mse": 82.18125121080213,
 

## Второй этап: взаимодействие с BatchNorm-архитектурой
Другой агент нашёл более устойчивую MLP: два скрытых слоя 128, каждый Linear → BatchNorm → ReLU → Dropout(0.4). Проверяем на ней те же 32 головы и функции потерь. Три лидера подтверждаем на seeds17,83. Итого 64 конфигурации выхода/архитектуры. Затем отбираем три лучшие по среднему MSE трёх разбиений; это отдельные модели, ансамбля нет.


In [12]:
# После независимого архитектурного поиска переносим все 32 головы на BN128.
bn_configs = [dict(c, id="bn_"+c["id"], backbone="batchnorm") for c in configs]
bn_results = []
for config in bn_configs:
    bn_results.append(run_trial(config, 42, stage="screen"))
    RESULT_PATH.write_text(json.dumps(results+confirmation+bn_results, indent=2))
print("BN screening:")
for r in sorted(bn_results, key=lambda r:r["mse"]):
    print(r["config"]["id"], round(r["mse"], 6), "epoch", r["best_epoch"])
bn_leader_ids = [r["config"]["id"] for r in sorted(bn_results, key=lambda r:r["mse"])][:3]
bn_leader_configs = [next(c for c in bn_configs if c["id"] == name) for name in bn_leader_ids]
bn_confirmation = []
for seed in [17, 83]:
    for config in bn_leader_configs:
        bn_confirmation.append(run_trial(config, seed, stage="confirmation"))
        RESULT_PATH.write_text(json.dumps(results+confirmation+bn_results+bn_confirmation, indent=2))
all_runs = results+confirmation+bn_results+bn_confirmation
extended_summary = []
for config in [baseline_config]+leader_configs+bn_leader_configs:
    runs = [r for r in all_runs if r["config"]["id"] == config["id"]]
    assert len(runs) == 3
    baseline_runs = {r["seed"]:r for r in all_runs if r["config"]["id"] == "baseline"}
    differences = [r["mse"]-baseline_runs[r["seed"]]["mse"] for r in runs]
    extended_summary.append(dict(id=config["id"], mean_mse=sum(r["mse"] for r in runs)/3,
                                 mean_delta=sum(differences)/3, seeds=[r["seed"] for r in runs],
                                 mse=[r["mse"] for r in runs], best_epochs=[r["best_epoch"] for r in runs]))
extended_summary.sort(key=lambda r:r["mean_mse"])
print("Все подтверждённые кандидаты:", json.dumps(extended_summary, indent=2))
(OUTPUT_DIR / "summary.json").write_text(json.dumps(extended_summary, indent=2))
selected_ids = [r["id"] for r in extended_summary if r["id"] != "baseline"][:3]
leader_configs = [next(c for c in configs+bn_configs if c["id"] == name) for name in selected_ids]
print("Final refit candidates:", selected_ids)


screen: bn_baseline, seed=42, MSE=80.990101, RMSE=8.999450, epoch=18, 1.3s
screen: bn_huber_d0.75, seed=42, MSE=82.529271, RMSE=9.084562, epoch=18, 1.2s
screen: bn_huber_d1.5, seed=42, MSE=81.372919, RMSE=9.020694, epoch=20, 1.4s
screen: bn_huber_d2.5, seed=42, MSE=80.817126, RMSE=8.989835, epoch=18, 1.2s
screen: bn_huber_d4, seed=42, MSE=81.046916, RMSE=9.002606, epoch=18, 1.3s
screen: bn_bounded_m0, seed=42, MSE=81.021217, RMSE=9.001179, epoch=18, 1.3s
screen: bn_bounded_m5, seed=42, MSE=81.145079, RMSE=9.008056, epoch=18, 1.3s
screen: bn_bounded_m15, seed=42, MSE=81.308178, RMSE=9.017105, epoch=18, 1.4s
screen: bn_upper_softplus, seed=42, MSE=80.495903, RMSE=8.971951, epoch=14, 1.3s
screen: bn_gaussian_m0_n1, seed=42, MSE=80.806250, RMSE=8.989230, epoch=37, 2.8s
screen: bn_gaussian_m1_n0.25, seed=42, MSE=80.153098, RMSE=8.952826, epoch=20, 1.8s
screen: bn_gaussian_m1_n1, seed=42, MSE=80.014832, RMSE=8.945101, epoch=24, 2.1s
screen: bn_dist_s0.6_ce0.1, seed=42, MSE=80.742619, RMSE=8.

## Третий этап: более сильная регуляризация распределения
В независимом поиске regularization более сильная конфигурация BN128/ReLU/Dropout0.55/weight_decay0.5 показала устойчивость по трём development-разбиениям. Проверяем на ней ширину мягкой метки, относительный вес CE и чистую CE. Также уточняем вес CE на исходном BN-backbone и проверяем weight_decay0.1. Это 28 дополнительных конфигураций; их три лучших повторно проверяем на seeds17,83. Всего 92 конфигурации и 112 запусков. Audit на всех этапах закрыт.


In [13]:
# Третий этап: сильнее регуляризованная BN-архитектура, найденная другим агентом.
refine_configs = []
for sigma in [0.25, 0.6, 2, 5]:
    for aw in [0.25, 1, 2, 4]:
        refine_configs.append(dict(common, id=f"reg55_wd5_s{sigma}_ce{aw}", kind="distribution",
            backbone="batchnorm", bn_dropout=0.55, weight_decay=0.5, bins=91, sigma=sigma,
            prior=True, mse_weight=1, aux_weight=aw))
for sigma in [0.6, 2, 5]:
    refine_configs.append(dict(common, id=f"reg55_wd5_purece_s{sigma}", kind="distribution",
        backbone="batchnorm", bn_dropout=0.55, weight_decay=0.5, bins=91, sigma=sigma,
        prior=True, mse_weight=0, aux_weight=1))
for aw in [0.25, 0.5, 2, 4]:
    refine_configs.append(dict(common, id=f"bn_s0.6_refine_ce{aw}", kind="distribution",
        backbone="batchnorm", bn_dropout=0.4, bins=91, sigma=0.6, prior=True,
        mse_weight=1, aux_weight=aw))
for sigma in [0.6, 2]:
    for aw in [1, 2]:
        refine_configs.append(dict(common, id=f"reg55_wd1_s{sigma}_ce{aw}", kind="distribution",
            backbone="batchnorm", bn_dropout=0.55, weight_decay=0.1, bins=91, sigma=sigma,
            prior=True, mse_weight=1, aux_weight=aw))
refine_configs.append(dict(common, id="reg55_wd5_mse", kind="mse", backbone="batchnorm",
                           bn_dropout=0.55, weight_decay=0.5))
refine_results = []
for config in refine_configs:
    refine_results.append(run_trial(config, 42, stage="screen"))
    RESULT_PATH.write_text(json.dumps(all_runs+refine_results, indent=2))
print("Refinement screening:")
for r in sorted(refine_results, key=lambda r:r["mse"]):
    print(r["config"]["id"], round(r["mse"],6), "epoch", r["best_epoch"])
refine_leader_ids = [r["config"]["id"] for r in sorted(refine_results,key=lambda r:r["mse"])][:3]
refine_leader_configs = [next(c for c in refine_configs if c["id"] == name) for name in refine_leader_ids]
refine_confirmation=[]
for seed in [17,83]:
    for config in refine_leader_configs:
        refine_confirmation.append(run_trial(config, seed, stage="confirmation"))
        RESULT_PATH.write_text(json.dumps(all_runs+refine_results+refine_confirmation, indent=2))
all_runs = all_runs+refine_results+refine_confirmation
all_configs = configs+bn_configs+refine_configs
complete_ids = [c["id"] for c in all_configs if len([r for r in all_runs if r["config"]["id"]==c["id"]]) == 3]
final_summary=[]
for name in complete_ids:
    runs=[r for r in all_runs if r["config"]["id"]==name]
    baseline_runs={r["seed"]:r for r in all_runs if r["config"]["id"]=="baseline"}
    differences=[r["mse"]-baseline_runs[r["seed"]]["mse"] for r in runs]
    final_summary.append(dict(id=name, mean_mse=sum(r["mse"] for r in runs)/3,
                             mean_delta=sum(differences)/3,seeds=[r["seed"] for r in runs],
                             mse=[r["mse"] for r in runs],best_epochs=[r["best_epoch"] for r in runs]))
final_summary.sort(key=lambda r:r["mean_mse"])
print("Итоговая таблица:",json.dumps(final_summary,indent=2))
(OUTPUT_DIR/"summary.json").write_text(json.dumps(final_summary,indent=2))
selected_ids=[r["id"] for r in final_summary if r["id"]!="baseline"][:3]
leader_configs=[next(c for c in all_configs if c["id"]==name) for name in selected_ids]
print("Выбраны для final refit:",selected_ids)
print("Уникальных конфигураций:", len(all_configs), "; обучений с подтверждениями:",len(all_runs))


screen: reg55_wd5_s0.25_ce0.25, seed=42, MSE=80.215620, RMSE=8.956317, epoch=13, 1.9s
screen: reg55_wd5_s0.25_ce1, seed=42, MSE=79.358150, RMSE=8.908319, epoch=18, 2.2s
screen: reg55_wd5_s0.25_ce2, seed=42, MSE=79.339922, RMSE=8.907296, epoch=20, 2.3s
screen: reg55_wd5_s0.25_ce4, seed=42, MSE=79.397339, RMSE=8.910518, epoch=34, 3.3s
screen: reg55_wd5_s0.6_ce0.25, seed=42, MSE=80.380513, RMSE=8.965518, epoch=13, 1.7s
screen: reg55_wd5_s0.6_ce1, seed=42, MSE=79.600127, RMSE=8.921890, epoch=18, 1.9s
screen: reg55_wd5_s0.6_ce2, seed=42, MSE=79.431385, RMSE=8.912429, epoch=20, 2.1s
screen: reg55_wd5_s0.6_ce4, seed=42, MSE=79.466518, RMSE=8.914400, epoch=31, 2.9s
screen: reg55_wd5_s2_ce0.25, seed=42, MSE=80.470320, RMSE=8.970525, epoch=13, 1.6s
screen: reg55_wd5_s2_ce1, seed=42, MSE=79.886632, RMSE=8.937932, epoch=13, 1.7s
screen: reg55_wd5_s2_ce2, seed=42, MSE=79.602252, RMSE=8.922009, epoch=18, 2.0s
screen: reg55_wd5_s2_ce4, seed=42, MSE=79.794117, RMSE=8.932755, epoch=18, 2.1s
screen: reg

## Проверка собственного оптимизатора на итоговой модели
Помимо численных unit-проверок AdamW повторяем лучший seed42 запуск, меняя только реализацию оптимизатора. Сравниваем лучшие эпохи, MSE и все validation-предсказания.


In [14]:
# Полное парное сравнение оптимизаторов на лучшей конфигурации, seed42.
custom_check = run_trial(leader_configs[0], 42, stage="optimizer_check", optimizer_kind="custom")
builtin_check = next(r for r in all_runs if r["config"]["id"]==leader_configs[0]["id"] and r["seed"]==42)
name = leader_configs[0]["id"]
a = torch.load(OUTPUT_DIR / f"{name}_seed42.pt", weights_only=False)
b = torch.load(OUTPUT_DIR / f"{name}_seed42_custom.pt", weights_only=False)
max_prediction_delta = float((a["valid_prediction"]-b["valid_prediction"]).abs().max())
torch.testing.assert_close(a["valid_prediction"], b["valid_prediction"], rtol=0, atol=1e-4)
optimizer_check = dict(id=name, builtin_mse=builtin_check["mse"], custom_mse=custom_check["mse"],
                       max_prediction_delta=max_prediction_delta,
                       builtin_epoch=builtin_check["best_epoch"], custom_epoch=custom_check["best_epoch"])
print("Сравнение CustomAdamW / torch.optim.AdamW:", optimizer_check)


optimizer_check: reg55_wd1_s0.6_ce2, seed=42, MSE=79.307344, RMSE=8.905467, epoch=20, 1.9s
Сравнение CustomAdamW / torch.optim.AdamW: {'id': 'reg55_wd1_s0.6_ce2', 'builtin_mse': 79.30734411031717, 'custom_mse': 79.30734411031717, 'max_prediction_delta': 0.0, 'builtin_epoch': 20, 'custom_epoch': 20}


## Финальные CSV
Для всех 9 альтернатив, подтверждённых по трём разбиениям, выполняем воспроизводимый refit на всех 14 000 строках и сохраняем отдельные CSV. Три лидера помечены в manifest. Число эпох — медиана лучших эпох; cosine horizon160 сохраняется. Каждый CSV содержит одну сеть, усреднения прогнозов нет. Test применяется только дляtransform/predict. ID и исходный порядок сохранены, форматid,year подтверждён успешной отправкой исходногоCSV пользователем.


In [15]:
submission_dir = DATA_DIR / "Submission"
submission_dir.mkdir(exist_ok=True)
final_artifacts = []
# Воспроизводим все подтверждённые варианты, отмечая три главных кандидата.
submission_configs = [next(c for c in all_configs if c["id"] == row["id"])
                      for row in final_summary if row["id"] != "baseline"]
for config in submission_configs:
    runs = [r for r in all_runs if r["config"]["id"] == config["id"]]
    final_epochs = int(statistics.median(r["best_epoch"] for r in runs))
    seed_everything(42)
    pp = FeaturePreprocessor(config.get("preprocessing", "standard")).fit(X)
    xt = pp.transform(X)
    target_scaler = TargetScaler(y)
    loader = make_loader(xt, target_scaler.encode(y), 42)
    model = ObjectiveNetwork(config, y)
    optimizer = build_optimizer(model, "custom", config["lr"], config["weight_decay"])
    history = []
    for epoch in range(1, final_epochs+1):
        model.train()
        for group in optimizer.param_groups:
            group["lr"] = learning_rate(config, epoch)
        total_loss = total_grad = 0.0
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            pred, raw = model(xb)
            loss = model.loss(pred, raw, yb)
            loss.backward()
            grad = nn.utils.clip_grad_norm_(model.parameters(), 5.0, error_if_nonfinite=True)
            optimizer.step()
            total_loss += float(loss.detach())*len(xb)
            total_grad += float(grad)*len(xb)
        history.append(dict(epoch=epoch, loss=total_loss/len(y), grad_norm=total_grad/len(y)))
    prediction = objective_predict(model, pp.transform(X_test))
    assert prediction.shape == (6000,) and torch.isfinite(prediction).all()
    submission = pd.DataFrame({"id": test_ids.to_numpy(), "year": prediction.numpy()})
    assert submission["id"].equals(test_ids.reset_index(drop=True))
    path = submission_dir / f'task4_objectives_{config["id"]}.csv'
    submission.to_csv(path, index=False, float_format="%.6f")
    roundtrip = pd.read_csv(path)
    assert list(roundtrip.columns) == ["id", "year"] and len(roundtrip) == 6000
    assert roundtrip["id"].equals(test_ids.reset_index(drop=True))
    torch.testing.assert_close(torch.tensor(roundtrip["year"].to_numpy()), prediction.double(), rtol=0, atol=1e-6)
    torch.save(dict(state=model.state_dict(), config=config, epochs=final_epochs,
                    prediction=prediction, target_mean=model.mean, target_std=model.std,
                    history=history), OUTPUT_DIR / f'final_{config["id"]}.pt')
    row = next(row for row in final_summary if row["id"] == config["id"])
    artifact = dict(id=config["id"], config=dict(config), refit_epochs=final_epochs,
                    path=str(path), sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                    validation_mean_mse=row["mean_mse"],
                    seed42_mse=next(r["mse"] for r in runs if r["seed"] == 42),
                    validation_mse={str(r["seed"]):r["mse"] for r in runs},
                    top3=config["id"] in selected_ids,
                    min_year=float(prediction.min()), max_year=float(prediction.max()))
    final_artifacts.append(artifact)
    print(f'Submission {config["id"]}: meanMSE={row["mean_mse"]:.6f}, '
          f'epochs={final_epochs}, SHA256={artifact["sha256"]}', flush=True)
assert len(list(submission_dir.glob("task4_objectives_*.csv"))) == len(final_artifacts)
(OUTPUT_DIR / "final_artifacts.json").write_text(json.dumps(final_artifacts, indent=2))
Path("/tmp/task4_objectives_manifest.json").write_text(json.dumps(final_artifacts, indent=2))
report = dict(results=all_runs, summary=final_summary, manifest=final_artifacts, optimizer_check=optimizer_check,
              top3=selected_ids, audit_evaluated=False, unique_configs=len(all_configs),
              validation_runs=len(all_runs), final_refits=len(final_artifacts))
Path("/tmp/task4_objectives_report.json").write_text(json.dumps(report, indent=2))
print("Финальные кандидаты:", selected_ids)
print("ВсегоCSV:",len(final_artifacts),"; audit не оценивался")


Submission reg55_wd1_s0.6_ce2: meanMSE=78.757411, epochs=31, SHA256=a1b684188bde0829dbb5192a1144bba887d20a3588760930b4484b16e72d8935
Submission reg55_wd5_s0.25_ce2: meanMSE=78.804686, epochs=28, SHA256=2ab3e5accf2203ec0a95d2c7674f198504318d9edb2cc7320248e487def0db19
Submission reg55_wd1_s0.6_ce1: meanMSE=78.816272, epochs=28, SHA256=a35137b51c872e12e194c8d2119aa31787c0547893db53e5a7ab90f903886574
Submission bn_dist_pure_ce_p1: meanMSE=78.934148, epochs=23, SHA256=a66fa3d560dc862a3868ba44028a0ac51990078368f610dff67436a7d721a4ce
Submission bn_dist_s0.6_ce1: meanMSE=79.370422, epochs=14, SHA256=f352ef250aaf530f12e668fcf07871a4077b203bda2962a3ba1c4f4c232e73ae
Submission bn_gaussian_m1_n1: meanMSE=80.526779, epochs=24, SHA256=9a4bd0ea8a28da76bb0bd567a385ffd41f327cb4956da05bcafc47d77d5940fa
Submission dist_s0.6_ce1: meanMSE=81.485166, epochs=4, SHA256=d47e19bc5a02a5c1f8cfb1c0b80a0f75bd7895e850d59a58855833aa48aa8305
Submission cdf_aux30: meanMSE=82.181251, epochs=3, SHA256=6a9d1f364d5abc1183a

## Границы вывода
Выбор из многих гипотез по validation может давать оптимистическую оценку. Три development-разбиения пересекаются, поэтому это проверка устойчивости, а не три независимых теста. До фиксации кандидатов audit оставался закрытым; диагностическая оценка замороженных моделей приведена в приложении. Отбор окончательного файла требует общей проверки кандидатов; public leaderboard не должен становиться заменой validation.


## Приложение: однократная диагностика на audit

После завершения всех волн поиска зафиксированы три лучших варианта objectives и pure-CE как отличающаяся постановка. По отдельному разрешению родительского агента открываем audit ровно для этих четырёх моделей. Никакие новые настройки, эпохи или способы обработки после этого не подбираются. Исходный baseline-audit был известен до текущего поиска; это диагностическая оценка замороженных моделей, а не новая выборка для подбора.

Оцениваются checkpoints, обученные на 9800 строках и выбранные по validation 2100, с train-only предобработкой. Final CSV относятся к другим, заново обученным на 14 000 строках моделям. Audit нельзя трактовать как точную будущую оценку этих CSV или score Kaggle. При загрузкеcheckpoint явно восстанавливаются mean/std из metadata, а low/high — конструктором по исходным train_idx. До audit проверяем точное совпадение восстановленных validation-предсказаний.


In [16]:
# Кандидаты зафиксированы до открытия audit; этот блок не выбирает новые настройки.
audit_candidate_ids = ["reg55_wd1_s0.6_ce2", "reg55_wd5_s0.25_ce2",
                       "reg55_wd1_s0.6_ce1", "bn_dist_pure_ce_p1"]
audit_results = []
for name in audit_candidate_ids:
    checkpoint_path = OUTPUT_DIR / f"{name}_seed42.pt"
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    fit_idx = checkpoint["train_idx"].cpu().numpy()
    holdout_idx = checkpoint["valid_idx"].cpu().numpy()
    assert not (set(fit_idx) & set(audit_idx))
    audit_preprocessing = FeaturePreprocessor(checkpoint["config"].get("preprocessing", "standard")).fit(X[fit_idx])
    # Конструктор получает именно обучающие targets: восстановлены также low/high.
    audit_model = ObjectiveNetwork(checkpoint["config"], y[fit_idx])
    audit_model.load_state_dict(checkpoint["state"])
    audit_model.mean = checkpoint["target_mean"]
    audit_model.std = checkpoint["target_std"]
    restored_valid = objective_predict(audit_model, audit_preprocessing.transform(X[holdout_idx]))
    torch.testing.assert_close(restored_valid, checkpoint["valid_prediction"], rtol=0, atol=0)
    prediction = objective_predict(audit_model, audit_preprocessing.transform(X[audit_idx]))
    metrics = regression_metrics(prediction, y[audit_idx])
    prediction_path = OUTPUT_DIR / f"audit_{name}.pt"
    torch.save(dict(id=name, row_indices=torch.tensor(audit_idx), prediction=prediction,
                    target=y[audit_idx], config=checkpoint["config"], **metrics), prediction_path)
    result = dict(id=name, seed=42, train_rows=len(fit_idx), audit_rows=len(audit_idx),
                  best_epoch=checkpoint["best_epoch"], config=checkpoint["config"],
                  checkpoint_path=str(checkpoint_path), prediction_path=str(prediction_path), **metrics)
    audit_results.append(result)
    print(f'{name:28s} MSE={metrics["mse"]:.6f} RMSE={metrics["rmse"]:.6f} MAE={metrics["mae"]:.6f}')
Path("/tmp/task4_objectives_audit.json").write_text(json.dumps(audit_results, indent=2))
report = json.loads(Path("/tmp/task4_objectives_report.json").read_text())
report["audit_evaluated"] = True
report["audit_used_for_selection"] = False
report["audit"] = audit_results
Path("/tmp/task4_objectives_report.json").write_text(json.dumps(report, indent=2))
print("Параметры, число эпох final refit и CSV после audit не менялись.")


reg55_wd1_s0.6_ce2           MSE=82.820659 RMSE=9.100586 MAE=6.242702
reg55_wd5_s0.25_ce2          MSE=83.248220 RMSE=9.124046 MAE=6.253326
reg55_wd1_s0.6_ce1           MSE=82.453495 RMSE=9.080391 MAE=6.275577
bn_dist_pure_ce_p1           MSE=81.551842 RMSE=9.030606 MAE=6.136077
Параметры, число эпох final refit и CSV после audit не менялись.
